# Pandas Essentials — AI Engineering Interview Prep

Covers: groupby, merge, time series, apply, performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option('display.max_columns', 20)
print("pandas:", pd.__version__)

In [ ]:
# Create sample DataFrames
n = 1000
df = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=n, freq='D'),
    'user_id': np.random.choice([f'U{i:03d}' for i in range(50)], n),
    'product': np.random.choice(['Widget', 'Gadget', 'Doohickey', 'Thingamajig'], n),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Sports'], n),
    'revenue': np.random.exponential(scale=100, size=n).round(2),
    'quantity': np.random.randint(1, 20, n),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
})
# Introduce some missing values
df.loc[np.random.choice(df.index, 50), 'revenue'] = np.nan
print(df.shape)
df.head()

## 1. GroupBy

In [ ]:
# Basic groupby
print("=== Revenue by category ===")
print(df.groupby('category')['revenue'].agg(['mean', 'sum', 'count', 'std']).round(2))

print("\n=== Multi-key groupby ===")
print(df.groupby(['category', 'region'])['revenue'].mean().unstack().round(2))

In [ ]:
# Custom aggregations with agg
agg_result = df.groupby('category').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    n_transactions=('revenue', 'count'),
    total_qty=('quantity', 'sum'),
    unique_users=('user_id', 'nunique'),
).round(2)
agg_result

In [ ]:
# Transform: add group-level stats to each row
df['category_mean'] = df.groupby('category')['revenue'].transform('mean')
df['pct_of_category'] = df['revenue'] / df['category_mean'] * 100
df[['category', 'revenue', 'category_mean', 'pct_of_category']].dropna().head(8)

## 2. Merge / Join

In [ ]:
users = pd.DataFrame({
    'user_id': [f'U{i:03d}' for i in range(50)],
    'age': np.random.randint(18, 65, 50),
    'plan': np.random.choice(['free', 'basic', 'premium'], 50)
})

# Inner merge
merged = df.merge(users, on='user_id', how='left')
print(f"After merge: {merged.shape}")

# Merge stats per plan
merged.groupby('plan')['revenue'].agg(['mean', 'count']).round(2)

## 3. Time Series

In [ ]:
ts = df.set_index('date')['revenue'].resample('W').sum()  # weekly totals

# Rolling / expanding
ts_df = ts.to_frame('revenue')
ts_df['rolling_4w'] = ts_df['revenue'].rolling(4).mean()
ts_df['expanding_mean'] = ts_df['revenue'].expanding().mean()
ts_df['pct_change'] = ts_df['revenue'].pct_change() * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
ts_df[['revenue', 'rolling_4w']].plot(ax=ax1, title='Weekly Revenue + 4-week Rolling Mean')
ts_df['pct_change'].plot(ax=ax2, title='Week-over-Week % Change', color='orange')
ax2.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Date features extraction
df['year']      = df['date'].dt.year
df['month']     = df['date'].dt.month
df['dayofweek'] = df['date'].dt.dayofweek
df['quarter']   = df['date'].dt.quarter
df['is_weekend'] = df['date'].dt.dayofweek >= 5

print("Weekend vs weekday avg revenue:")
print(df.groupby('is_weekend')['revenue'].mean().round(2))

## 4. Apply & Custom Functions

In [ ]:
# apply row-wise
def revenue_tier(row):
    if pd.isna(row['revenue']):
        return 'unknown'
    elif row['revenue'] >= 200:
        return 'high'
    elif row['revenue'] >= 75:
        return 'mid'
    return 'low'

df['tier'] = df.apply(revenue_tier, axis=1)
print(df['tier'].value_counts())

# Faster with np.select (vectorized alternative)
conditions = [df['revenue'] >= 200, df['revenue'] >= 75]
choices    = ['high', 'mid']
df['tier_fast'] = np.select(conditions, choices, default='low')
print("\nUsing np.select (faster):")
print(df['tier_fast'].value_counts())

## 5. Missing Values & Data Quality

In [ ]:
print("Missing values:")
print(df.isnull().sum())

# Fill strategies
df['revenue_filled_mean'] = df['revenue'].fillna(df['revenue'].mean())
df['revenue_filled_ffwd'] = df['revenue'].ffill()
df['revenue_filled_cat']  = df.groupby('category')['revenue'].transform(lambda x: x.fillna(x.median()))

print(f"\nAfter fill — remaining NaN: {df['revenue_filled_cat'].isna().sum()}")

## Interview Tips

- **groupby + transform** adds group-level stats back to each row without reducing shape.
- **merge types**: inner (intersection), left (all left), outer (union) — same as SQL joins.
- **resample** requires DatetimeIndex. Set with `df.set_index('date')` first.
- **apply(axis=1)** is slow for large data — prefer vectorized ops (`np.where`, `np.select`, `.str`, `.dt`).
- **Category dtype**: `df['col'].astype('category')` saves memory for low-cardinality string columns.
- **chained assignment** (`df[mask]['col'] = val`) doesn't work — use `df.loc[mask, 'col'] = val`.